Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

try:
    df = pd.read_csv('StudentsPerformance.csv')
    print("Data Loaded Successfully.")
except FileNotFoundError:
    print("Error: 'StudentsPerformance.csv' not found. Please ensure it is in the same folder.")

print(f"Data Shape: {df.shape}")
print("\nData Types & Memory Usage:")
df.info()

print("\nDescriptive Statistics:")
print(df.describe())

print("\nCardinality")
print(df.nunique())

Missing Value Analysis

In [ ]:
missing_percent = df.isnull().mean() * 100

print("Missing Value Percentages:")
print(missing_percent)

plt.figure(figsize=(10, 5))
plt.bar(missing_percent.index, missing_percent.to_numpy(), color='pink')
plt.title('Percentage of Missing Values')
plt.ylabel('Percentage %')
plt.xlabel('Columns')
plt.xticks(rotation=45)
plt.ylim(0, max(1, missing_percent.max() + 1))
plt.savefig('missing_values.png', bbox_inches='tight')
print("Figure saved: 'missing_values.png'")
plt.show()

for col in df.columns:
    if df[col].isnull().sum() > 0:
        if df[col].dtype == 'object':
            df[col] = df[col].fillna(df[col].mode()[0])
        else:
            df[col] = df[col].fillna(df[col].median())
        print(f"Imputed missing values in column: {col}")
    else:
        print(f"No missing values to impute in: {col}")

Column Selection And Filtering

In [ ]:
df1 = df.loc[(df['math score'] > 80) & (df['reading score'] > 80) & (df['writing score'] > 80)]
print(df1)

df2 = df.iloc[:10, :4]
print(df2)

df3 = df[df['parental level of education'].str.contains('associate')]
print(df3)

df4 = df[df['race/ethnicity'].str.startswith('group A')]
print(df4)

df5 = df[df['parental level of education'].str.endswith('high school')]
print(df5)

df6 = df[(df['reading score'] >= 60) | (df['writing score'] >= 60)]
print(df6)

Feature Engineering

In [ ]:
print("Before Feature Engineering:\n")
print(df[['math score', 'reading score', 'writing score', 'lunch']].head())

df['average_score'] = (df['math score'] + df['reading score'] + df['writing score']) / 3

df['grade'] = pd.cut(df['average_score'],
                     bins=[0, 60, 70, 80, 90, 100],
                     labels=['F', 'D', 'C', 'B', 'A'])

df['math_status'] = np.where(df['math score'] >= 60, 'Pass', 'Fail')

df['lunch_short'] = df['lunch'].str.extract(r'^(\w+)')

df['score_spread'] = df[['math score', 'reading score', 'writing score']].max(axis=1) - \
                     df[['math score', 'reading score', 'writing score']].min(axis=1)

print("\nAfter Feature Engineering:\n")
print(df[['math score', 'average_score', 'grade', 'math_status', 'lunch', 'lunch_short']].head())

Grouping and Pivot Tables

In [ ]:
df7 = df.groupby('race/ethnicity')['average_score'].agg(['mean', 'min', 'max', 'std', 'count'])
print("Ethnicity Grouping:\n")
print(df7)

df8 = df.pivot_table(values='math score',
                       index='parental level of education',
                       columns='gender',
                       aggfunc='mean')

print("\nMath Score Pivot:\n")
print(df8)

df7['mean'].plot(kind='bar', color='blue', title='Average Score by Ethnicity')
plt.ylabel('Score')
plt.savefig('grouped.png')
plt.show()

df8.plot(kind='bar', figsize=(10,5), title='Math Score by Parent Education & Gender')
plt.ylabel('Math Score')
plt.xticks(rotation=45)
plt.savefig('pivot.png')
plt.show()

Sorting and Ranking

In [ ]:
df9 = df.sort_values(by=['math score', 'reading score'], ascending=[False, False])

df['rank_dense'] = df['average_score'].rank(method='dense', ascending=False)
df['rank_average'] = df['average_score'].rank(method='average', ascending=False)
df['rank_first'] = df['average_score'].rank(method='first', ascending=False)

print("Top 10 Students:\n")
print(df9[['gender', 'math score', 'reading score', 'average_score']].head(10))

print("\nBottom 10 Students:\n")
print(df9[['gender', 'math score', 'reading score', 'average_score']].tail(10))

String Operations

In [ ]:
df10 = df.select_dtypes(include=['object']).columns

for col in df10:
    df[col] = df[col].str.strip().str.lower()

df['gender_num'] = df['gender'].str.extract(r'^(\w)')

df['race_num'] = df['race/ethnicity'].str.extract(r'group (\w)')

df['edu_type'] = df['parental level of education'].str.extract(r'(\w+)$')

df['lunch_num'] = df['lunch'].str.extract(r'^(\w+)')

df['prep_num'] = df['test preparation course'].str.extract(r'(^.{4})')

print("Regex:")
print("\nGender:")
print(df['gender_num'].value_counts())

print("\nRace:")
print(df['race_num'].value_counts())

print("\nEducation:")
print(df['edu_type'].value_counts())

print("\nLunch:")
print(df['lunch_num'].value_counts())

print("\nPrep Course:")
print(df['prep_num'].value_counts())

Data Export For Cleaned Dataset

In [ ]:
df.to_csv('StudentsPerformance_Cleaned.csv', index=False)

with pd.ExcelWriter('Summary_Tables.xlsx') as writer:
    df7.to_excel(writer, sheet_name='Grouped_Stats')